In [8]:
%load_ext autoreload
%autoreload 2

In [9]:
import os
import json
from datetime import datetime
from parse_process import parse_case_files, parse_metadata_files, parse_oneliner_files

In [15]:
cases_dir = os.listdir('./key_cases_scrapped_full/')

In [21]:
def load_txt(file_path, mode):
    if mode == 'read':
        with open(file_path, 'r') as f:
            return f.read()
    elif mode == 'readlines':
        with open(file_path, 'r') as f:
            return f.readlines()
    else:
        raise ValueError('Invalid mode')

def save_json(file_path, data):
    with open(file_path, 'w') as f:
        json.dump(data, f)

In [16]:
len(cases_dir)

149

In [33]:
all_one_liners = []
for case in cases_dir:
    case_i = os.path.join('./key_cases_scrapped_full/', case)
    
    files = sorted(os.listdir(case_i))
    case_data = {}
    # add case id
    case_data['itemid'] = case
    # parse one-liner
    case_i_oneliner = load_txt(case_i + f'/{files[1]}', 'readlines')
    extracted_oneliner = case_i_oneliner#parse_oneliner_files(case_i_oneliner)
    case_data['one-liner'] = extracted_oneliner
    # extract year
    year = files[0].split('-')[0]
    case_data['year'] = year

    assert len(case_data['one-liner']) > 0, f'No one-liner found for {case}'

    all_one_liners.append(case_data)

In [34]:
len(all_one_liners)

149

In [35]:
# from list of dicts to dataframe
import pandas as pd
df = pd.DataFrame(all_one_liners)

In [36]:
df

,itemid,one-liner,year
0,001-202524,"[9 Key cases 2020 Cases by Article\n, Fa...",2020
1,001-187540,"[HEAVIER PENALTY | RETROACTIVITY\n, Subsequent...",2018
2,001-220484,[No evidence showing a real risk of a sentence...,2022
3,001-233206,"[ARTICLE 34\n, LOCUS STANDI\n, VICTIM\n, Victi...",2024
4,001-216179,"[ARTICLE 10\n, FREEDOM OF EXPRESSION\n, No leg...",2022
...,...,...,...
144,001-234468,"[Article 6 § 2\n, PRESUMPTION OF INNOCENCE\n, ...",2024
145,001-229129,"[5\n, Key cases 2023\n, ARTICLE 4\n, TRAFFICKI...",2023
146,001-224435,"[POSITIVE OBLIGATIONS\n, Segregation, humiliat...",2023
147,001-182243,"[Article 6 § 3 (c)\n, DEFENCE THROUGH LEGAL AS...",2018


In [37]:
itemids = list(df['itemid'].values)

In [43]:
not_exist = []
for itemid in itemids:
    data_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed'
    file_name = itemid + '.json'
    data_path = os.path.join(data_path, file_name)
    # check if file exists
    if not os.path.exists(data_path):
        #print(f'{data_path} does not exist')
        not_exist.append(itemid)

In [45]:
f' {len(itemids)} files, where {len(itemids) - len(not_exist)} exists and {len(not_exist)} do not exist'

' 149 files, where 64 exists and 85 do not exist'

In [75]:
eval_df = pd.read_csv('/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_1.csv')
eval_ids = list(eval_df['file_path'].values) 
eval_ids = [i.split('.')[0] for i in eval_ids]

len(eval_ids)

125

In [82]:
# intersection of itemids and eval_ids = how many eval_ids are have one liners
intersection = list(set(itemids).intersection(set(eval_ids)))
len(intersection)

64

In [83]:
intersection

['001-220484',
 '001-181866',
 '001-194618',
 '001-206153',
 '001-219943',
 '001-200657',
 '001-187930',
 '001-203840',
 '001-187391',
 '001-200442',
 '001-181821',
 '001-220960',
 '001-203213',
 '001-187605',
 '001-206369',
 '001-187802',
 '001-202524',
 '001-186135',
 '001-209373',
 '001-203165',
 '001-181591',
 '001-207173',
 '001-186216',
 '001-203825',
 '001-206582',
 '001-219333',
 '001-183395',
 '001-180486',
 '001-201646',
 '001-187932',
 '001-218424',
 '001-189781',
 '001-210363',
 '001-202621',
 '001-188985',
 '001-209520',
 '001-214330',
 '001-181273',
 '001-203169',
 '001-187188',
 '001-186767',
 '001-187540',
 '001-189426',
 '001-194307',
 '001-184525',
 '001-185306',
 '001-202532',
 '001-182243',
 '001-200344',
 '001-182452',
 '001-218933',
 '001-205222',
 '001-206515',
 '001-203645',
 '001-194523',
 '001-188268',
 '001-219067',
 '001-201342',
 '001-204603',
 '001-201353',
 '001-204993',
 '001-181789',
 '001-205536',
 '001-203503']

In [85]:
df[df['itemid'].isin(intersection)]

,itemid,one-liner,year
0,001-202524,"[9 Key cases 2020 Cases by Article\n, Fa...",2020
1,001-187540,"[HEAVIER PENALTY | RETROACTIVITY\n, Subsequent...",2018
2,001-220484,[No evidence showing a real risk of a sentence...,2022
6,001-194307,"[EFFECTIVE INVESTIGATION\n, Alleged failure to...",2019
7,001-206582,"[TRIBUNAL ESTABLISHED BY LAW\n, Participation ...",2020
...,...,...,...
140,001-186216,"[RESPECT FOR PRIVATE LIFE\n, Dismissal of judg...",2018
141,001-188985,"[ARTICLE 14\n, DISCRIMINATION (ARTICLE 1 OF PR...",2018
142,001-203645,"[5 Key cases 2020 Cases by Article\n, AR...",2020
147,001-182243,"[Article 6 § 3 (c)\n, DEFENCE THROUGH LEGAL AS...",2018


In [88]:
def append_to_txt(file_path, data):
    with open(file_path, 'a') as f:
        f.write(data)

In [96]:
for row in df[df['itemid'].isin(intersection)].iterrows():
    item_id = row[1]['itemid']
    one_liner = ''.join(row[1]['one-liner'])
    string = f'{item_id}\n\n{one_liner}\n'
    string += '------------------------------------------------------------------------\n'
    append_to_txt('/Users/ahmed/Desktop/msc-24/TND/key_cases_scraping_parsing_processing/one-liners/easy-to-scrap.txt', string)

In [77]:
# cases that are not included in eval_ids
diff = list(set(itemids) - (set(eval_ids)))
len(diff)

85

In [69]:
# cases in eval_ids missing one-liner
missing_one_liner = list(set(eval_ids) - set(intersection))
len(missing_one_liner)

61

In [70]:
64 + 61

125

In [71]:
missing_one_liner

['001-209069',
 '001-183947',
 '001-172701',
 '001-198760',
 '001-171804',
 '001-191587',
 '001-210077',
 '001-208053',
 '001-210463',
 '001-176769',
 '001-211592',
 '001-175646',
 '001-172913',
 '001-198811',
 '001-207953',
 '001-175667',
 '001-208326',
 '001-193494',
 '001-170436',
 '001-169054',
 '001-177299',
 '001-172963',
 '001-207633',
 '001-170054',
 '001-207360',
 '001-197098',
 '001-177349',
 '001-169662',
 '001-170347',
 '001-175180',
 '001-175659',
 '001-177406',
 '001-211125',
 '001-180442',
 '001-177429',
 '001-178753',
 '001-208877',
 '001-175121',
 '001-170388',
 '001-192616',
 '001-174422',
 '001-210078',
 '001-170359',
 '001-169663',
 '001-214433',
 '001-177082',
 '001-172327',
 '001-172660',
 '001-197253',
 '001-191276',
 '001-207757',
 '001-208279',
 '001-213908',
 '001-192210',
 '001-189641',
 '001-186828',
 '001-173805',
 '001-179219',
 '001-210874',
 '001-213827',
 '001-170663']

In [73]:
df.to_csv('one_liners.csv', index=False)

In [ ]:
dfp = pd.read_csv('one_liners.csv')

In [78]:
diff

['001-203370',
 '001-224629',
 '001-228016',
 '001-189019',
 '001-207927',
 '001-220073',
 '001-195544',
 '001-225209',
 '001-193543',
 '001-237239',
 '001-217565',
 '001-225645',
 '001-223675',
 '001-234982',
 '001-225655',
 '001-194051',
 '001-187507',
 '001-180276',
 '001-217436',
 '001-235475',
 '001-229366',
 '001-218516',
 '001-202468',
 '001-220616',
 '001-205361',
 '001-222654',
 '001-223259',
 '001-217488',
 '001-219984',
 '001-216707',
 '001-222660',
 '001-217963',
 '001-222410',
 '001-233174',
 '001-231227',
 '001-203885',
 '001-204996',
 '001-229726',
 '001-235426',
 '001-206897',
 '001-222889',
 '001-205509',
 '001-222780',
 '001-229133',
 '001-224435',
 '001-233381',
 '001-236065',
 '001-216872',
 '001-216179',
 '001-225231',
 '001-216400',
 '001-225652',
 '001-207115',
 '001-184438',
 '001-229376',
 '001-227636',
 '001-234468',
 '001-229129',
 '001-203534',
 '001-224928',
 '001-220007',
 '001-233214',
 '001-233206',
 '001-223924',
 '001-218512',
 '001-234499',
 '001-2192